In [11]:
import numpy as np
import struct
from array import array
from os.path  import join

%matplotlib inline
import random
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

In [2]:
# local to this repo

from data import build_dataloaders
from model import ConvModel
from util import test_loss, test_accuracy
from optim import GradSignOptimizer    # my custom optimizer

In [7]:
cross_entropy = nn.CrossEntropyLoss()

In [8]:
# Just find out how many params
# We will re-initialize the model in-loop for reproducibility

model = ConvModel()
sum([p.numel() for p in model.parameters()])

66954

In [19]:
# Constant across experimental runs
n_batches = 1000


# Stochastic gradient descent

In [ ]:
# training loop

# a good range for learning rate with SGD
for lr in [0.003, 0.01, 0.03, 0.1]:


    torch.manual_seed(1)
    model = ConvModel()
    
    train_loader, test_loader = build_dataloaders(seed=1)

    # For reproducibility (see the GradSign section below for why we need this)
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            
            # Uncomment to see partial training progress
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            loss.backward()
    
            with torch.no_grad():
                for p in model.parameters():
                    p -= lr * p.grad
            model.zero_grad()
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")


tensor([9, 3, 0, 1, 2, 4, 6, 8, 1, 1, 2, 3, 0, 7, 0, 0, 1, 3, 9, 4, 1, 1, 2, 9,
        0, 0, 1, 4, 6, 2, 1, 3])
Parameter containing:
tensor([[[[-1.1664e-02, -4.4616e-03, -1.1190e-02],
          [-2.1722e-02,  2.4591e-02, -2.0070e-02],
          [-3.3103e-02,  1.3368e-02, -4.2592e-02]],

         [[-1.2576e-02,  3.0112e-02,  2.2022e-02],
          [ 5.0166e-02, -4.0190e-02,  5.1233e-04],
          [-1.7401e-02, -3.2958e-02, -3.3149e-02]],

         [[ 1.4845e-02,  6.1651e-03, -3.0078e-02],
          [-4.1534e-02, -4.8821e-02,  5.7751e-02],
          [-1.0044e-02, -1.3626e-02, -2.1668e-02]],

         ...,

         [[ 1.3534e-02, -1.8116e-02,  5.2515e-02],
          [ 1.5043e-02, -4.0045e-02, -3.2795e-02],
          [ 3.6947e-02,  2.6913e-02,  4.6009e-02]],

         [[-4.5264e-03,  4.1214e-02,  2.9297e-02],
          [ 1.8223e-02, -1.3638e-02,  5.6804e-02],
          [ 1.1924e-02, -1.5204e-02, -8.3778e-04]],

         [[ 5.7927e-02,  3.9577e-02, -4.3668e-03],
          [ 5.7769e-02, 

# GradSign, my custom optimizer

In [ ]:
#training loop

for lr in [0.001, 0.01, 0.1]:
    
    torch.manual_seed(1)
    model = ConvModel()
    train_loader, test_loader = build_dataloaders(seed=1)
    
    optim = GradSignOptimizer(model.named_parameters(), lr=lr)
    model.train()

    # For reproducibility (because optim.__init__() called the torch RNG)
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            # Uncomment to see partial training progress
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")



In [16]:
test_accuracy(model, test_loader)

Computed test accuracy over 10000 items


0.944

In [22]:
#training loop

for lr in [0.001, 0.01, 0.1]:
    
    torch.manual_seed(1)
    model = ConvModel()
    train_loader, test_loader = build_dataloaders(seed=1)
    
    optim = Adam(model.parameters(), lr=lr)
    model.train()
    
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            if i % 250 == 0 or i == n_batches - 1:
                print(f"Loss at step {i:4} is {loss.item():0.3f}")
                print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")



Loss at step    0 is 2.435
Computed test loss over 10000 items
Test loss at step    0 is 2.306
Loss at step  250 is 0.102
Computed test loss over 10000 items
Test loss at step  250 is 0.086
Loss at step  500 is 0.093
Computed test loss over 10000 items
Test loss at step  500 is 0.083
Loss at step  750 is 0.006
Computed test loss over 10000 items
Test loss at step  750 is 0.050
Loss at step  999 is 0.143
Computed test loss over 10000 items
Test loss at step  999 is 0.052
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.984
Loss at step    0 is 2.435
Computed test loss over 10000 items
Test loss at step    0 is 2.306
Loss at step  250 is 0.167
Computed test loss over 10000 items
Test loss at step  250 is 0.173
Loss at step  500 is 0.128
Computed test loss over 10000 items
Test loss at step  500 is 0.089
Loss at step  750 is 0.038
Computed test loss over 10000 items
Test loss at step  750 is 0.095
Loss at step  999 is 0.067
Computed test loss over 10000 items